In [4]:
import json
import re
from pathlib import Path
from bs4 import BeautifulSoup

ROOT_DIR = Path("Site_Suggestions")
OUT_FILE = Path("Site_Suggestions.json")

def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

messages = []
seen = set()

for page_file in ROOT_DIR.glob("thread_*/page_*.html"):
    with open(page_file, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f.read(), "lxml")

    for post in soup.select("article.b-post.js-post, article.b-post"):
        author_el = post.select_one("a.user-link.js-username, a.user-link")
        time_el = post.select_one("footer.b-post--footer time, time")
        content_el = post.select_one("div.js-content")

        if not content_el:
            continue

        author = clean_text(author_el.get_text(" ", strip=True)) if author_el else None
        date = clean_text(time_el.get_text(" ", strip=True)) if time_el else None
        text = clean_text(content_el.get_text("\n", strip=True))

        key = (author or "", date or "", text)
        if key in seen:
            continue
        seen.add(key)

        messages.append({
            "author": author,
            "date": date,
            "text": text
        })

with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(messages, f, ensure_ascii=False, indent=2)

print(f"Saved {len(messages)} messages to {OUT_FILE}")

Saved 31742 messages to Site_Suggestions.json
